In [10]:
"""
Script de prueba sencilla: Configuración de fuerza dedo a dedo.
Hereda de la librería base para usar su conexión nativa.
"""

import time
from inspire_rh56dftp.client import InspireHand

# Definimos el registro Modbus para configurar la fuerza manualmente.
# Si en el nuevo mapa de memoria de la RH56DFTP la dirección cambió, 
# solo debes actualizar este número (ej. 1504, 1006, etc.)
FORCE_SET = 1498

class InspireHandAvanzada(InspireHand):
    def send_force(self, force_values):
        """
        Envía 6 valores de fuerza límite a la mano.
        Sigue la misma lógica de envío sincrónico de send_raw() del client.py.
        """
        if self.dead:
            return False
            
        # Aseguramos que los valores estén en un rango seguro (0 a 1500)
        clamped = [max(0, min(1500, int(v))) for v in force_values]
        
        try:
            # Usamos la función interna de la librería que formatea la trama y drena el ACK
            self._write_and_ack(FORCE_SET, clamped)
            return True
        except Exception as e:
            print(f"Error enviando fuerzas: {e}")
            return False

def main():
    IP_MANO = "192.168.123.211"
    print(f"Conectando a la mano en {IP_MANO}...")
    
    # Usamos nuestra nueva clase que ya incluye la función de fuerza
    mano = InspireHandAvanzada(ip=IP_MANO)

    # 1. Definir fuerzas: [Meñique, Anular, Corazón, Índice, Flexión pulgar, Rot. pulgar]
    # Valores más bajos para que puedas sentir que la fuerza limitadora funciona.
    fuerzas_prueba = [500, 500, 500, 500, 500, 500]

    # 2. Definir una postura para que puedas resistir el movimiento con tus dedos
    angulos_prueba = [500, 500, 500, 500, 500, 1000]

    print(f"Aplicando fuerza límite de: {fuerzas_prueba}")
    exito_fuerza = mano.send_force(fuerzas_prueba)
    
    time.sleep(2)
    if exito_fuerza:
        time.sleep(0.1) # Pequeña pausa para procesar Modbus
        
        print(f"Cerrando dedos a la posición: {angulos_prueba}")
        mano.send_raw(angulos_prueba)
        
        print("Mano moviéndose... tienes 5 segundos para probar la resistencia de los dedos.")
        time.sleep(3)
        
        print("Abriendo la mano para finalizar...")
        mano.send_raw([1000,1000,1000,1000,150,1000])
        time.sleep(1)
    else:
        print("No se pudo configurar la fuerza.")

    #mano.close()
    print("Prueba finalizada. Conexión cerrada.")

if __name__ == "__main__":
    main()

Conectando a la mano en 192.168.123.211...
Aplicando fuerza límite de: [500, 500, 500, 500, 500, 500]
Cerrando dedos a la posición: [500, 500, 500, 500, 500, 1000]
Mano moviéndose... tienes 5 segundos para probar la resistencia de los dedos.
Abriendo la mano para finalizar...
Prueba finalizada. Conexión cerrada.
